# 230968078 - Ishan Suryawanshi - Lab 6

### Exercise 1

In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler

In [2]:
SEED = 42
rng  = np.random.default_rng(SEED)

In [3]:
data    = load_breast_cancer()
X_raw   = data.data
y       = data.target
scaler  = StandardScaler()
X       = scaler.fit_transform(X_raw)
n_feat  = X.shape[1]
feat_names = data.feature_names

In [4]:
print("=" * 62)
print("  Hill-Climbing Feature Selection — Breast Cancer Dataset")
print("=" * 62)
print(f"  Samples : {X.shape[0]}   |   Features : {n_feat}   |   Classes : 2\n")

  Hill-Climbing Feature Selection — Breast Cancer Dataset
  Samples : 569   |   Features : 30   |   Classes : 2



In [5]:
def evaluate(state: np.ndarray) -> float:
    """5-fold CV accuracy of Logistic Regression on the selected features."""
    selected = np.where(state == 1)[0]
    if len(selected) == 0:
        return 0.0
    clf = LogisticRegression(max_iter=1000, random_state=SEED)
    scores = cross_val_score(clf, X[:, selected], y, cv=5, scoring="accuracy")
    return scores.mean()

In [6]:
def get_neighbours(state: np.ndarray):
    """Yield all single-bit-flip neighbours (n neighbours total)."""
    neighbours = []
    for i in range(len(state)):
        neighbour = state.copy()
        neighbour[i] ^= 1
        neighbours.append(neighbour)
    return neighbours

In [7]:
def hill_climbing(initial_state: np.ndarray, verbose: bool = False):
    current       = initial_state.copy()
    current_score = evaluate(current)
    steps         = 0

    while True:
        neighbours    = get_neighbours(current)
        best_nb       = max(neighbours, key=evaluate)
        best_nb_score = evaluate(best_nb)

        if best_nb_score <= current_score:
            break

        current       = best_nb
        current_score = best_nb_score
        steps        += 1

        if verbose:
            n_sel = current.sum()
            print(f"    step {steps:3d} | acc={current_score:.4f} | "
                  f"features selected={n_sel}")

    return current, current_score, steps

In [8]:
N_RESTARTS  = 10
results     = []

In [9]:
for run in range(1, N_RESTARTS + 1):
    while True:
        init_state = rng.integers(0, 2, size=n_feat)
        if init_state.sum() > 0:
            break

    init_score            = evaluate(init_state)
    best_state, best_acc, n_steps = hill_climbing(init_state)

    results.append({
        "run"         : run,
        "init_acc"    : init_score,
        "final_acc"   : best_acc,
        "steps"       : n_steps,
        "n_features"  : int(best_state.sum()),
        "state"       : best_state,
    })

    print(f"  {run:>3}  {init_score:>10.4f}  {best_acc:>10.4f}  "
          f"{n_steps:>6}  {int(best_state.sum()):>11}")

    1      0.9701      0.9807       3           19
    2      0.9561      0.9754       2           15
    3      0.9772      0.9824       2           16
    4      0.9701      0.9825       3           17
    5      0.9719      0.9807       2           16
    6      0.9684      0.9877       3           18
    7      0.9737      0.9824       3           15
    8      0.9631      0.9754       2           14
    9      0.9737      0.9824       4           14
   10      0.9543      0.9824       4           15


In [10]:
best   = max(results, key=lambda r: r["final_acc"])
worst  = min(results, key=lambda r: r["final_acc"])
mean   = np.mean([r["final_acc"] for r in results])
std    = np.std ([r["final_acc"] for r in results])

In [11]:
print("\n" + "=" * 62)
print("  Summary across all restarts")
print("=" * 62)
print(f"  Best  accuracy : {best['final_acc']:.4f}  (run {best['run']}, "
      f"{best['n_features']} features)")
print(f"  Worst accuracy : {worst['final_acc']:.4f}  (run {worst['run']}, "
      f"{worst['n_features']} features)")
print(f"  Mean  accuracy : {mean:.4f}  ±  {std:.4f}")

baseline_state = np.ones(n_feat, dtype=int)
baseline_acc   = evaluate(baseline_state)
print(f"\n  Baseline (all {n_feat} features) accuracy : {baseline_acc:.4f}")

best_features = feat_names[best["state"] == 1]
print(f"\n  Best feature subset ({best['n_features']} features):")
for i, f in enumerate(best_features, 1):
    print(f"    {i:2d}. {f}")

df = pd.DataFrame([{k: v for k, v in r.items() if k != "state"}
                   for r in results])
print("\n  Per-Run Results:")
print(df.to_string(index=False))
print("=" * 62)


  Summary across all restarts
  Best  accuracy : 0.9877  (run 6, 18 features)
  Worst accuracy : 0.9754  (run 2, 15 features)
  Mean  accuracy : 0.9812  ±  0.0034

  Baseline (all 30 features) accuracy : 0.9807

  Best feature subset (18 features):
     1. mean radius
     2. mean perimeter
     3. mean compactness
     4. mean concave points
     5. mean symmetry
     6. mean fractal dimension
     7. radius error
     8. texture error
     9. area error
    10. compactness error
    11. fractal dimension error
    12. worst texture
    13. worst perimeter
    14. worst area
    15. worst smoothness
    16. worst concavity
    17. worst symmetry
    18. worst fractal dimension

  Per-Run Results:
 run  init_acc  final_acc  steps  n_features
   1  0.970129   0.980686      3          19
   2  0.956063   0.975408      2          15
   3  0.977162   0.982425      2          16
   4  0.970144   0.982456      3          17
   5  0.971868   0.980671      2          16
   6  0.968390   0.987

### Exercise 2

In [12]:
SEED = 42
rng  = np.random.default_rng(SEED)

In [13]:
N_LOCATIONS = 20
AREA        = 100.0

depot     = np.array([50.0, 50.0])
locations = rng.uniform(0, AREA, size=(N_LOCATIONS, 2))
all_pts   = np.vstack([depot, locations])

print("=" * 60)
print("  Simulated Annealing — Vehicle Routing Optimization")
print("=" * 60)
print(f"  Deliveries : {N_LOCATIONS}   |   Grid : {AREA}x{AREA}\n")

  Simulated Annealing — Vehicle Routing Optimization
  Deliveries : 20   |   Grid : 100.0x100.0



In [14]:
def route_distance(route: np.ndarray) -> float:
    """Total distance: depot → route[0] → … → route[-1] → depot."""
    full  = np.concatenate(([0], route, [0]))
    pts   = all_pts[full]
    diffs = np.diff(pts, axis=0)
    return float(np.sum(np.hypot(diffs[:, 0], diffs[:, 1])))

In [15]:
def swap_neighbour(route: np.ndarray) -> np.ndarray:
    nb   = route.copy()
    i, j = rng.choice(len(nb), size=2, replace=False)
    nb[i], nb[j] = nb[j], nb[i]
    return nb

In [16]:
def simulated_annealing(
    T_init   = 1000.0,
    T_min    = 1e-3,
    alpha    = 0.9998,
    max_iter = 200_000,
):
    current      = rng.permutation(np.arange(1, N_LOCATIONS + 1))
    current_cost = route_distance(current)
    best, best_cost = current.copy(), current_cost

    T, it = T_init, 0
    accepted_worse = 0

    while T > T_min and it < max_iter:
        neighbour      = swap_neighbour(current)
        neighbour_cost = route_distance(neighbour)
        delta          = neighbour_cost - current_cost

        if delta < 0 or rng.random() < np.exp(-delta / T):
            current, current_cost = neighbour, neighbour_cost
            if delta > 0:
                accepted_worse += 1

        if current_cost < best_cost:
            best, best_cost = current.copy(), current_cost

        T  *= alpha
        it += 1

    return best, best_cost, it, accepted_worse

In [17]:
def nearest_neighbour():
    unvisited = list(range(1, N_LOCATIONS + 1))
    route, pos = [], 0
    while unvisited:
        nearest = min(unvisited,
                      key=lambda j: np.linalg.norm(all_pts[pos] - all_pts[j]))
        route.append(nearest)
        pos = nearest
        unvisited.remove(nearest)
    return np.array(route)

In [18]:
nn_route  = nearest_neighbour()
nn_dist   = route_distance(nn_route)
rand_dist = route_distance(rng.permutation(np.arange(1, N_LOCATIONS + 1)))

In [19]:
best_route, best_dist, total_it, accepted_worse = simulated_annealing()

improvement = (nn_dist - best_dist) / nn_dist * 100

print(f"  {'Method':<32} {'Distance':>10}")
print(f"  {'-'*44}")
print(f"  {'Random route':<32} {rand_dist:>10.2f}")
print(f"  {'Nearest-Neighbour greedy':<32} {nn_dist:>10.2f}")
print(f"  {'Simulated Annealing (best)':<32} {best_dist:>10.2f}")
print(f"\n  Iterations completed  : {total_it:,}")
print(f"  Worse moves accepted  : {accepted_worse:,}")
print(f"  Improvement vs greedy : {improvement:+.1f}%")
print(f"\n  Best route found:")
print(f"  depot → {' → '.join(str(s-1) for s in best_route)} → depot")
print("=" * 60)

  Method                             Distance
  --------------------------------------------
  Random route                        1101.03
  Nearest-Neighbour greedy             400.39
  Simulated Annealing (best)           400.93

  Iterations completed  : 69,071
  Worse moves accepted  : 7,702
  Improvement vs greedy : -0.1%

  Best route found:
  depot → 12 → 10 → 0 → 9 → 1 → 11 → 15 → 3 → 6 → 5 → 2 → 14 → 19 → 18 → 4 → 16 → 7 → 17 → 13 → 8 → depot


### Exercise 3

In [20]:
MAP_SIZE     = 100.0
START        = np.array([0.0,   0.0 ])
GOAL         = np.array([100.0, 100.0])
N_WAYPOINTS  = 8

OBSTACLES = [
    (25, 30, 8),
    (50, 45, 10),
    (70, 60, 9),
    (40, 75, 7),
    (60, 20, 6),
]

In [21]:
POP_SIZE        = 100
N_GENERATIONS   = 300
TOURNAMENT_K    = 3
CROSSOVER_RATE  = 0.85
MUTATION_RATE   = 0.25
MUTATION_SIGMA  = 8.0
ELITISM         = 2
COLLISION_PEN   = 500.0
CONV_WINDOW     = 30

print("=" * 62)
print("  Genetic Algorithm — Drone Path Planning")
print("=" * 62)
print(f"  Map      : {MAP_SIZE}x{MAP_SIZE}   Start: {START}   Goal: {GOAL}")
print(f"  Obstacles: {len(OBSTACLES)}   Waypoints: {N_WAYPOINTS}")
print(f"  Pop size : {POP_SIZE}   Generations: {N_GENERATIONS}\n")

  Genetic Algorithm — Drone Path Planning
  Map      : 100.0x100.0   Start: [0. 0.]   Goal: [100. 100.]
  Obstacles: 5   Waypoints: 8
  Pop size : 100   Generations: 300



In [22]:
def clip_to_map(pts: np.ndarray) -> np.ndarray:
    return np.clip(pts, 0.0, MAP_SIZE)

In [23]:
def segment_hits_obstacle(p1, p2, cx, cy, r, samples=20):
    """Check if the line segment p1→p2 passes within radius r of (cx,cy)."""
    ts  = np.linspace(0, 1, samples)
    pts = p1 + np.outer(ts, p2 - p1)
    return np.any(np.hypot(pts[:, 0] - cx, pts[:, 1] - cy) < r)

In [24]:
def path_cost(chromosome: np.ndarray) -> tuple[float, int]:
    """
    Returns (total_distance, n_collisions).
    chromosome shape: (N_WAYPOINTS, 2)
    Full path: START → waypoints → GOAL
    """
    full = np.vstack([START, chromosome, GOAL])
    diffs = np.diff(full, axis=0)
    distance = float(np.sum(np.hypot(diffs[:, 0], diffs[:, 1])))

    collisions = 0
    for i in range(len(full) - 1):
        for (cx, cy, r) in OBSTACLES:
            if segment_hits_obstacle(full[i], full[i+1], cx, cy, r):
                collisions += 1

    return distance, collisions

In [25]:
def fitness(chromosome: np.ndarray) -> float:
    dist, cols = path_cost(chromosome)
    return dist + cols * COLLISION_PEN

In [26]:
def random_chromosome() -> np.ndarray:
    """Random waypoints uniformly distributed within map bounds."""
    return clip_to_map(rng.uniform(0, MAP_SIZE, size=(N_WAYPOINTS, 2)))

population = [random_chromosome() for _ in range(POP_SIZE)]

In [27]:
def tournament_select(pop, fitnesses, k=TOURNAMENT_K) -> np.ndarray:
    """Pick k random individuals, return the best."""
    idxs = rng.choice(len(pop), size=k, replace=False)
    best = idxs[np.argmin([fitnesses[i] for i in idxs])]
    return pop[best].copy()

In [28]:
def one_point_crossover(p1: np.ndarray, p2: np.ndarray) -> tuple:
    """Split at a random waypoint index and swap tails."""
    point = rng.integers(1, N_WAYPOINTS)
    c1 = np.vstack([p1[:point], p2[point:]])
    c2 = np.vstack([p2[:point], p1[point:]])
    return c1, c2

In [29]:
def mutate(chromosome: np.ndarray) -> np.ndarray:
    """Gaussian perturbation of each waypoint with probability MUTATION_RATE."""
    child = chromosome.copy()
    for i in range(N_WAYPOINTS):
        if rng.random() < MUTATION_RATE:
            child[i] += rng.normal(0, MUTATION_SIGMA, size=2)
    return clip_to_map(child)

In [30]:
best_overall      = None
best_fitness_ever = float("inf")
prev_best         = float("inf")
stagnation        = 0

In [31]:
for gen in range(1, N_GENERATIONS + 1):

    fitnesses = [fitness(c) for c in population]

    gen_best_idx = int(np.argmin(fitnesses))
    gen_best_fit = fitnesses[gen_best_idx]
    if gen_best_fit < best_fitness_ever:
        best_fitness_ever = gen_best_fit
        best_overall      = population[gen_best_idx].copy()

    if abs(prev_best - gen_best_fit) < 1e-4:
        stagnation += 1
    else:
        stagnation = 0
    prev_best = gen_best_fit

    if gen % 50 == 0 or gen == 1:
        dist, cols = path_cost(best_overall)
        print(f"  Gen {gen:>4} | best fitness={gen_best_fit:>8.2f} | "
              f"dist={dist:>7.2f} | collisions={cols}")

    if stagnation >= CONV_WINDOW:
        print(f"\n  [Early stop] No improvement for {CONV_WINDOW} generations.")
        break

    sorted_idx  = np.argsort(fitnesses)
    new_pop     = [population[i].copy() for i in sorted_idx[:ELITISM]]

    while len(new_pop) < POP_SIZE:
        p1 = tournament_select(population, fitnesses)
        p2 = tournament_select(population, fitnesses)

        if rng.random() < CROSSOVER_RATE:
            c1, c2 = one_point_crossover(p1, p2)
        else:
            c1, c2 = p1.copy(), p2.copy()

        new_pop.append(mutate(c1))
        if len(new_pop) < POP_SIZE:
            new_pop.append(mutate(c2))

    population = new_pop

  Gen    1 | best fitness=  857.83 | dist= 357.83 | collisions=1
  Gen   50 | best fitness=  146.11 | dist= 146.11 | collisions=0
  Gen  100 | best fitness=  144.41 | dist= 144.41 | collisions=0

  [Early stop] No improvement for 30 generations.


In [32]:
final_dist, final_cols = path_cost(best_overall)
straight_line          = float(np.linalg.norm(GOAL - START))

print("\n" + "=" * 62)
print("  Final Results")
print("=" * 62)
print(f"  Straight-line distance (no obstacles) : {straight_line:.2f}")
print(f"  Best evolved path distance            : {final_dist:.2f}")
print(f"  Obstacle collisions on best path      : {final_cols}")
print(f"  Path overhead vs straight line        : "
      f"{(final_dist - straight_line) / straight_line * 100:.1f}%")
print(f"\n  Best waypoint sequence:")
print(f"  START {tuple(START)}")
for i, wp in enumerate(best_overall, 1):
    print(f"  WP {i:>2}  ({wp[0]:6.2f}, {wp[1]:6.2f})")
print(f"  GOAL  {tuple(GOAL)}")
print("=" * 62)


  Final Results
  Straight-line distance (no obstacles) : 141.42
  Best evolved path distance            : 144.41
  Obstacle collisions on best path      : 0
  Path overhead vs straight line        : 2.1%

  Best waypoint sequence:
  START (np.float64(0.0), np.float64(0.0))
  WP  1  (  6.34,  11.11)
  WP  2  ( 18.94,  36.27)
  WP  3  ( 20.99,  39.39)
  WP  4  ( 23.05,  40.57)
  WP  5  ( 28.47,  43.96)
  WP  6  ( 39.43,  54.66)
  WP  7  ( 48.32,  61.58)
  WP  8  ( 89.42,  92.27)
  GOAL  (np.float64(100.0), np.float64(100.0))
